网格读取、几何结构生成、流场初始化

In [35]:
import meshreading as mr
import geometry as geo
import initialize as ini
import classconfig as cc
import solvesupple as ss

mr.read_mesh("fangdata.txt")
geo.geometry_main("output.txt")
ini.initialization_main()

tscell : cc.cell_class= cc.CellList[1][1]

successfully read fangdata.txt with 120 points.
i_total: 10, j_total: 12, meshcnt: 120


守恒量计算,计算的守恒量是$\boldsymbol{U} = (\rho, \rho u, \rho v, \rho E, \rho \tilde{\nu})$,使用`cc.cell_class.formvars`方法计算.


In [36]:
for i in range(1, cc.i_total,1):  # 一定要注意i的范围是什么
    for j in range(1,cc.j_total+1,1):
        cell : cc.cell_class = cc.CellList[i][j]
        cell.formvars()

# 测试tscell的守恒变量是否正确
# print(tscell.U)

基于CFL和谱半径近似的最小时间步长,公式为$\Delta t_{ij} = \frac{\text{CFL} \cdot V_{ij}}{|uA+vB| + |uC+vD| + c_{ij} \cdot (\sqrt{A^2+B^2} + \sqrt{C^2+D^2})}$,其中(A,B)和(C,D)为相邻两面法向的平均值.

In [37]:
import math
mintime = float('inf')

# 第一轮:计算各单元 localdt, 记录最小值
for j in range(1, cc.j_total + 1):
    jp1 = j + 1 if j < cc.j_total else 1          # 周向回绕
    for i in range(1, cc.i_total):
        cell : cc.cell_class= cc.CellList[i][j]
        A = 0.5 * (cc.Facelist_tau[i][j].nx+ cc.Facelist_tau[i+1][j].nx)
        B = 0.5 * (cc.Facelist_tau[i][j].ny+ cc.Facelist_tau[i+1][j].ny)
        C = 0.5 * (cc.FaceList_n[i][j].nx+ cc.FaceList_n[i][jp1].nx)
        D = 0.5 * (cc.FaceList_n[i][j].ny+ cc.FaceList_n[i][jp1].ny)
        E = abs(cell.u * A + cell.v * B)
        F = abs(cell.u * C + cell.v * D)
        G = math.sqrt(A * A + B * B)
        L = math.sqrt(C * C + D * D)
        cell.localdt = cc.CFL * cell.vol / (E + F + cell.c * (G + L))

        if cell.localdt < mintime:
            mintime = cell.localdt

# 第二轮:所有单元均使用全局最小时间步推进
for j in range(1, cc.j_total + 1):
    for i in range(1, cc.i_total):
        cc.CellList[i][j].dt = mintime

cc.totaltime += mintime
print("current time:", cc.totaltime,"min_time:", mintime)

current time: 0.00044076390910298765 min_time: 0.00044076390910298765


生成虚拟网格`Imagine_Mesh`,共有壁面虚拟网格、远场虚拟网格、成环虚拟网格三种类型

In [38]:
"""设置壁面虚拟网格,使用镜像法"""

for im in range(1, cc.IM + 1):
    ghost_row = [[]]                       # j=0 占位
    for j in range(1, cc.j_total + 1):
        gcell : cc.cell_class = cc.cell_class((cc.i_total + im - 1, j))

        # 标量: 从壁面直接复制
        gcell.rho = cc.CellList[1][j].rho
        gcell.p   = cc.CellList[1][j].p
        gcell.T   = cc.CellList[1][j].T
        gcell.E   = cc.CellList[1][j].E
        gcell.H   = cc.CellList[1][j].H
        gcell.c   = cc.CellList[1][j].c

        # 速度 / 湍流粘度: 取对应内层的相反数 (镜像反射)
        gcell.u     = -cc.CellList[im][j].u
        gcell.v     = -cc.CellList[im][j].v
        gcell.miubl = -cc.CellList[im][j].miubl
        gcell.ma = (math.sqrt(cc.CellList[im][j].u ** 2 +
                                cc.CellList[im][j].v ** 2) / cc.CellList[1][j].c)
        gcell.formvars()
        ghost_row.append(gcell)
    cc.CellList.append(ghost_row)

"""设置压力远场假想网格边界条件"""

for im in range(1, cc.IM + 1):
    ghost_row = [[]]             # j=0 占位
    for j in range(1, cc.j_total + 1):
        gcell = cc.cell_class((cc.i_total + im - 1, j))
        gcell.copy_flow_fields(cc.CellList[cc.i_total - 1][j])
        gcell.formvars()
        ghost_row.append(gcell)
    cc.CellList.append(ghost_row)

"""设置 O 型网格切割线两侧的周期假想网格 (j 方向周期边界)
左侧 ghost ← 右侧物理端 (j = j_total, j_total-1, ...)
右侧 ghost ← 左侧物理端 (j = 1, 2, ...)"""

for i in range(1, cc.i_total):
    # ── 左侧假想网格 ──
    for im in range(1, cc.IM + 1):
        gcell = cc.cell_class((i, cc.j_total + im))
        gcell.copy_flow_fields(cc.CellList[i][cc.j_total - im + 1])
        gcell.formvars()
        cc.CellList[i].append(gcell)

    # ── 右侧假想网格 ──
    for im in range(1, cc.IM + 1):
        gcell = cc.cell_class((i, cc.j_total + cc.IM + im))
        gcell.copy_flow_fields(cc.CellList[i][im])
        gcell.formvars()
        cc.CellList[i].append(gcell)

# print(vars(cc.CellList[cc.i_total+cc.IM-1][1]))

计算面上守恒量,通量的意思是经过某个网格单元四周边界流入/出的守恒量$\boldsymbol{U}$的净变化.

1. 先在每个面上准备面上的守恒量$\boldsymbol{U_f}$,其做法是$\boldsymbol{U_f} = \frac{\boldsymbol{U_1}+\boldsymbol{U_2}}{2}$.
2. 根据这个守恒量,变化出一种对流项$\boldsymbol{F}$
3. 把所有面通量汇总到单元中心,形成净通量残差$\boldsymbol{Q_c}$

In [ ]:
# 处理壁面处的面上守恒量,face_tau,由首层tau网格和第一层壁面虚拟网格平均
for j in range(1,cc.j_total+1):
    face : cc.face_class = cc.Facelist_tau[1][j]
    face.form_face_conserved(cc.CellList[1][j], cc.CellList[cc.i_total][j])

# 处理远场处的面上守恒量,face_tau,由最外层tau网格和第一层远场虚拟网格平均
for j in range(1,cc.j_total+1):
    face : cc.face_class = cc.Facelist_tau[cc.i_total][j]
    face.form_face_conserved(cc.CellList[cc.i_total-1][j], cc.CellList[cc.i_total+cc.IM][j])

# 处理左周期边界处的面上守恒量,face_n
for i in range(1,cc.i_total):
    face : cc.face_class = cc.FaceList_n[i][cc.j_total]
    face.form_face_conserved(cc.CellList[i][cc.j_total], cc.CellList[i][cc.j_total+1])
    
# 处理右周期边界处的面上守恒量,face_n
for i in range(1,cc.i_total):
    face : cc.face_class = cc.FaceList_n[i][1]
    face.form_face_conserved(cc.CellList[i][1], cc.CellList[i][cc.j_total+cc.IM+1])

# 处理正常地方的面上守恒量,时刻提醒自己face_tau是(i_total,j_total),face_n是(i_total-1,j_total)
for i in range(2,cc.i_total): # (1,j)和(i_total,j)的face通量已经处理好了
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.Facelist_tau[i][j]
        face.form_face_conserved(cc.CellList[i][j], cc.CellList[i-1][j])


for i in range(1,cc.i_total):
    for j in range(2,cc.j_total+1): # (i,1)和(i,j_total)的face通量已经处理好了
        face : cc.face_class = cc.FaceList_n[i][j]
        face.form_face_conserved(cc.CellList[i][j],cc.CellList[i][j-1])

# 现在开始构造通量项:
for i in range(1,cc.i_total+1):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.Facelist_tau[i][j]
        face.form_flux()

for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.FaceList_n[i][j]
        face.form_flux()


[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.49104641e+05 1.77835460e-06]
[0.00000000e+00 1.20076745e+00 8.14002693e+01 0.00000000e+00
 2.491046